# Perceptron e redes neurais simples

**Objetivo:** treinar um único neurônio (em PyTorch) num problema linearmente separável, ver que ele **falha** no XOR, e que uma rede com **uma camada oculta** resolve o XOR.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)
import torch

## 1. Um neurônio num problema separável

Um neurônio é `Linear(2,1)` seguido de sigmoide — literalmente uma regressão logística. Treinamos com o laço explícito de sempre.

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=200, centers=[[-2, -2], [2, 2]], cluster_std=1.2,
                  random_state=SEMENTE)
ent = torch.tensor(X, dtype=torch.float32)
alvo = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

torch.manual_seed(SEMENTE)
neuronio = torch.nn.Sequential(torch.nn.Linear(2, 1), torch.nn.Sigmoid())
custo_fn = torch.nn.BCELoss()
oti = torch.optim.SGD(neuronio.parameters(), lr=0.1)
for epoca in range(300):
    perda = custo_fn(neuronio(ent), alvo)
    oti.zero_grad(); perda.backward(); oti.step()
with torch.no_grad():
    acc = (((neuronio(ent) > 0.5).float() == alvo).float().mean()).item()
print("acuracia do neuronio (dados separaveis):", round(acc, 3))

In [ ]:
# fronteira de decisao do neuronio
passo = 0.05
gx, gy = np.meshgrid(np.arange(X[:,0].min()-1, X[:,0].max()+1, passo),
                     np.arange(X[:,1].min()-1, X[:,1].max()+1, passo))
with torch.no_grad():
    zz = neuronio(torch.tensor(np.c_[gx.ravel(), gy.ravel()], dtype=torch.float32)).numpy().reshape(gx.shape)
figura = go.Figure()
figura.add_trace(go.Contour(x=gx[0], y=gy[:,0], z=zz, showscale=False,
                            colorscale=[[0,"#dce7f4"],[1,"#f6dedb"]], opacity=0.6,
                            contours=dict(start=0.5, end=0.5, size=1, coloring="lines")))
figura.add_trace(go.Scatter(x=X[:,0], y=X[:,1], mode="markers",
                            marker=dict(color=y, colorscale="Bluered", size=6)))
figura.update_layout(title="Um neuronio: fronteira linear", height=380,
                     showlegend=False, margin=dict(l=10,r=10,t=50,b=10))
figura.show()

## 2. O XOR quebra o neurônio

No XOR, a classe é 1 quando as entradas **diferem**. Não há reta que separe — geramos uma versão ruidosa e treinamos o mesmo neurônio.

In [ ]:
rng = np.random.RandomState(SEMENTE)
centros = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
rotulos = np.array([0,1,1,0])   # XOR
Xx = np.repeat(centros, 60, axis=0) + rng.normal(0, 0.12, size=(240,2))
yy = np.repeat(rotulos, 60)
entx = torch.tensor(Xx, dtype=torch.float32)
alvox = torch.tensor(yy, dtype=torch.float32).reshape(-1,1)

torch.manual_seed(SEMENTE)
neuronio2 = torch.nn.Sequential(torch.nn.Linear(2,1), torch.nn.Sigmoid())
oti = torch.optim.Adam(neuronio2.parameters(), lr=0.05)
for epoca in range(400):
    perda = custo_fn(neuronio2(entx), alvox)
    oti.zero_grad(); perda.backward(); oti.step()
with torch.no_grad():
    acc1 = (((neuronio2(entx)>0.5).float()==alvox).float().mean()).item()
print("acuracia de UM neuronio no XOR:", round(acc1, 3), "(preso perto de 0.5-0.75)")

## 3. Uma camada oculta resolve

Agora `Linear(2,8) → ReLU → Linear(8,1) → Sigmoide`: a camada oculta cria representações que tornam o XOR separável. Mesmo laço, rede um pouco maior.

In [ ]:
torch.manual_seed(SEMENTE)
rede = torch.nn.Sequential(
    torch.nn.Linear(2, 8), torch.nn.ReLU(),
    torch.nn.Linear(8, 1), torch.nn.Sigmoid())
oti = torch.optim.Adam(rede.parameters(), lr=0.05)
for epoca in range(400):
    perda = custo_fn(rede(entx), alvox)
    oti.zero_grad(); perda.backward(); oti.step()
with torch.no_grad():
    acc2 = (((rede(entx)>0.5).float()==alvox).float().mean()).item()
print("acuracia de UM neuronio no XOR:", round(acc1, 3))
print("acuracia da rede com camada oculta:", round(acc2, 3))

## Exercício

Compare `acc1` e `acc2`. Por que acrescentar uma camada oculta (com ReLU) muda tão radicalmente o resultado no XOR?

<details><summary>Ver resposta</summary>

Um neurônio só traça **uma reta**, e o XOR não é linearmente separável — daí `acc1` travar longe de 100%. A camada oculta com ReLU cria **várias** fronteiras lineares e as combina de forma **não linear**, construindo uma representação em que as classes do XOR passam a ser separáveis pela camada final. É a não linearidade da camada oculta que quebra a limitação — `acc2` chega perto de 100%.

</details>